# TranAD

**Auteur**: `Jakob De Vreese`  
**Bachelorproef**: Overdraagbare unsupervised anomaliedetectie bij hybride HVAC-systemen  
**Academiejaar**: `2025-2026`

## Doel

Implementatie van **TranAD** (Tuli et al., 2022) voor anomaliedetectie in hybride HVAC-systemen. TranAD is een encoder-decoder Transformer met **twee-fasen adversariele training** en focus-score conditionering: de reconstructiefout uit fase 1 dient als prior om het model in fase 2 te focussen op de moeilijkste tijdstappen.

## Stappenplan

| Stap | Inhoud |
|---|---|
| 1 | Data preprocessing (feature reductie, normalisatie, windowing) |
| 2 | Architectuur (Encoder, Window Encoder, twee Decoders) |
| 3 | Initiele training (baseline sanity check) |
| 4 | Evaluatie van het baseline model (strategievergelijking, scorekaart, eindvisualisatie) |
| 5 | Hyperparameter tuning (KerasTuner Bayesiaanse optimalisatie) |
| 6 | Evaluatie van het beste model (strategievergelijking, scorekaart, eindvisualisatie) |
| 7 | Basis inferentie (laden van opgeslagen model) |

In [ ]:
import numpy as np
import pandas as pd
import json
import joblib
import os
from datetime import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, precision_recall_fscore_support,
    accuracy_score, balanced_accuracy_score, matthews_corrcoef,
    roc_auc_score, average_precision_score, fbeta_score,
)
from scipy.stats import genpareto

import tensorflow as tf
from tensorflow.keras import layers, Model
import keras_tuner as kt

import matplotlib.pyplot as plt
import seaborn as sns

import random
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)


## Parameters

Pas hier de configuratie aan om te schakelen naar een ander gebouw of andere instellingen. De rest van de notebook leest automatisch van deze variabelen.

In [ ]:
# ── Configureerbare parameters ─────────────────────────────────────────────
GEBOUW         = 'dunant1'  # Gebouwnaam (bestandsprefiks voor CSV en opgeslagen artefacten)
WINDOW_SIZE    = 144        # Vensterbreedte: 144 × 10 min = 24 uur
CORR_THRESHOLD = 0.97       # Correlatiedrempel voor feature-reductie
MIN_RUN        = 3          # Minimale alarmlengte voor run-length filter (tijdstappen)
NUM_EPOCHS     = 100        # Maximum epochs voor (her)training

## Stap 1 - Data Preprocessing

Zelfde voorbereiding als de Encoder-Only Transformer voor directe vergelijkbaarheid:

1. **Data inlezen**
2. **Feature reductie** (correlatie > 95% verwijderen)
3. **Data opsplitsen** (70 / 15 / 15)
4. **Normalisatie** (Z-score, gefitst op train)
5. **Windowing** - 3D-tensoren (samples, window_size, features)

### 1.1 Dataset inlezen

In [ ]:
url  = f'../02_eda_en_ground_truth/processed/{GEBOUW}_train.csv'
data = pd.read_csv(url)
data.head()

In [ ]:
print(data.shape)

### 1.2 Feature reductie

Een feature per paar met correlatie **> 95%** wordt verwijderd - zelfde drempel als de encoder-only implementatie. Dit reduceert redundantie en verlaagt de modelcomplexiteit.

In [ ]:
# Correlatie berekend uitsluitend op de trainset (eerste 70%) — geen lekkage van val/test
n_train = int(len(data) * 0.70)
train_for_corr = data.iloc[:n_train].drop(columns=["timestamp"])

corr_matrix = train_for_corr.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [col for col in upper.columns if any(upper[col] > CORR_THRESHOLD)]
kept_features = [col for col in data.columns if col not in to_drop and col != "timestamp"]

print(f'Behouden features: {len(kept_features)} (drempel: >{CORR_THRESHOLD})')

os.makedirs('tranad', exist_ok=True)
with open(f'tranad/features_{GEBOUW}.json', 'w') as f:
    json.dump(kept_features, f)

data_filtered = data[kept_features]

### 1.3 Data opsplitsen

In [ ]:
n = len(data_filtered)
train_df = data_filtered.iloc[:int(n * 0.70)]
val_df   = data_filtered.iloc[int(n * 0.70):int(n * 0.85)]
test_df  = data_filtered.iloc[int(n * 0.85):]

### 1.4 Normalisatie (Z-score)

De scaler wordt **alleen** op `train_df` gefitst en daarna op val en test toegepast - geen datalekkage.

In [ ]:
scaler = StandardScaler()
scaler.fit(train_df)

train_scaled = scaler.transform(train_df)
val_scaled   = scaler.transform(val_df)
test_scaled  = scaler.transform(test_df)

joblib.dump(scaler, f'tranad/scaler_{GEBOUW}.pkl')
print('Scaler opgeslagen.')

### 1.5 Windowing (3D tensor)

TranAD gebruikt een schuivend venster van **`WINDOW_SIZE`** tijdstappen (standaard 144 = 1 dag bij 10-minuten intervallen). Elk venster W_t = {x_{t-K+1},...,x_t} dient als reconstructiedoel (par. 3.2 van de paper).

In [ ]:
def create_windows(data_array, window_size=144, step=1):
    windows = []
    for i in range(0, len(data_array) - window_size + 1, step):
        windows.append(data_array[i:i + window_size])
    return np.array(windows)

X_train = create_windows(train_scaled, WINDOW_SIZE)
X_val   = create_windows(val_scaled,   WINDOW_SIZE)
X_test  = create_windows(test_scaled,  WINDOW_SIZE)

print(f'X_train: {X_train.shape}  X_val: {X_val.shape}  X_test: {X_test.shape}')

## Stap 2 - Architectuur

TranAD (Figuur 1, Tuli et al. 2022) bestaat uit:

- **Encoder E**: verwerkt venster W geconditioneerd op focus-score F; leert de globale temporele context
- **Twee decoders** D1 en D2: produceren reconstructies O1, O2 en O2-hat via cross-attention met de encoder

### Twee-fasen doorstroom

**Fase 1** - F = 0 (nulmatrix):
O1, O2 <- D1(E(W, 0)), D2(E(W, 0))

**Fase 2** - F = |O1 - W| (focus score die afwijkingen uitvergroot):
O2-hat <- D2(E(W, |O1 - W|))

**Anomaliescore** (Algoritme 2, lijn 4):
s = 0.5 * ||O1 - W|| + 0.5 * ||O2-hat - W||

### 2.1 Encoder Laag (Post-LN, conform paper)

In [ ]:
class TransformerEncoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_expansion=4, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = tf.keras.Sequential([
            layers.Dense(d_model * ffn_expansion, activation='relu'),
            layers.Dropout(dropout_rate),
            layers.Dense(d_model),
        ])
        self.dropout1 = layers.Dropout(dropout_rate)
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=False):
        attn = self.mha(x, x, training=training)
        attn = self.dropout1(attn, training=training)
        out1 = self.layernorm1(x + attn)
        ffn  = self.ffn(out1, training=training)
        return self.layernorm2(out1 + ffn)

### 2.2 Decoder Laag

Elke decoder bevat drie sub-lagen: self-attention, cross-attention met de encoder-output, en een point-wise FFN. De cross-attention laat de decoder de gecodeerde tijdreekscontext bevragen om de reconstructie te genereren.

In [ ]:
class TransformerDecoderLayer(layers.Layer):
    def __init__(self, d_model, num_heads, ffn_expansion=4, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.self_attn  = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.cross_attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads)
        self.ffn = tf.keras.Sequential([
            layers.Dense(d_model * ffn_expansion, activation='relu'),
            layers.Dropout(dropout_rate),
            layers.Dense(d_model),
        ])
        self.dropout1 = layers.Dropout(dropout_rate)
        self.dropout2 = layers.Dropout(dropout_rate)
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.ln3 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, memory, training=False):
        attn1 = self.self_attn(x, x, training=training)
        out1  = self.ln1(x + self.dropout1(attn1, training=training))
        attn2 = self.cross_attn(query=out1, key=memory, value=memory, training=training)
        out2  = self.ln2(out1 + self.dropout2(attn2, training=training))
        return self.ln3(out2 + self.ffn(out2, training=training))

### 2.3 TranAD Model - Twee-fasen adversariele training

`TranAD` implementeert Algoritme 1 van de paper:

- **Fase 1**: focus = nul -> O1, O2 berekenen
- **Fase 2**: focus = |O1 - W| -> O2-hat berekenen (adversarieel)
- **Verliescombinatie**: L = alpha * L_fase1 + (1-alpha) * L_fase2

De weging alpha schuift gradueel van alpha_start naar alpha_end - zodat de adversariele fase 2 meer gewicht krijgt naarmate fase 1 betrouwbaarder is. De `TranADPhaseScheduler` callback past alpha bij aan het begin van elke epoch.

In [ ]:
class TranAD(Model):
    def __init__(self, num_features, d_model=64, num_heads=4, num_layers=1,
                 ffn_expansion=4, dropout_rate=0.1,
                 alpha_start=0.6, alpha_end=0.1, total_epochs=50,
                 window_size=144, **kwargs):
        super().__init__(**kwargs)
        self.alpha_start   = float(alpha_start)
        self.alpha_end     = float(alpha_end)
        self.total_epochs  = int(total_epochs)
        self.current_epoch = tf.Variable(0.0, trainable=False, dtype=tf.float32)

        self.input_proj = layers.Dense(d_model)
        self.focus_proj = layers.Dense(d_model)
        self.pos_enc = self.add_weight(
            shape=(1, window_size, d_model), initializer='random_normal',
            trainable=True, name='pos_enc')

        self.encoder_layers = [
            TransformerEncoderLayer(d_model, num_heads, ffn_expansion, dropout_rate)
            for _ in range(num_layers)
        ]
        self.decoder1 = TransformerDecoderLayer(d_model, num_heads, ffn_expansion, dropout_rate)
        self.decoder2 = TransformerDecoderLayer(d_model, num_heads, ffn_expansion, dropout_rate)
        self.out_proj1 = layers.Dense(num_features)
        self.out_proj2 = layers.Dense(num_features)

        self.loss_tracker   = tf.keras.metrics.Mean(name='loss')
        self.phase1_tracker = tf.keras.metrics.Mean(name='phase1_loss')
        self.phase2_tracker = tf.keras.metrics.Mean(name='phase2_loss')

    @property
    def metrics(self):
        return [self.loss_tracker, self.phase1_tracker, self.phase2_tracker]

    def set_epoch(self, epoch):
        self.current_epoch.assign(float(epoch))

    def _alpha(self):
        denom = tf.cast(tf.maximum(self.total_epochs - 1, 1), tf.float32)
        t = tf.clip_by_value(self.current_epoch / denom, 0.0, 1.0)
        return tf.clip_by_value(
            self.alpha_start + (self.alpha_end - self.alpha_start) * t, 0.0, 1.0)

    def _encode_decode(self, x, focus, training=False):
        z = self.input_proj(x) + self.focus_proj(focus) + self.pos_enc
        for enc in self.encoder_layers:
            z = enc(z, training=training)
        tgt  = self.input_proj(x) + self.pos_enc
        out1 = self.out_proj1(self.decoder1(tgt, z, training=training))
        out2 = self.out_proj2(self.decoder2(tgt, z, training=training))
        return out1, out2

    def call(self, x, training=False):
        # Fase 1: focus = 0
        o1, _ = self._encode_decode(x, tf.zeros_like(x), training=training)
        # Fase 2: focus = |O1 - W|
        _, o2_hat = self._encode_decode(x, tf.abs(o1 - x), training=training)
        return o1, o2_hat

    def _compute_loss(self, x, training):
        # Fase 1 forward
        o1, o2 = self._encode_decode(x, tf.zeros_like(x), training=training)
        l_o1 = tf.reduce_mean(tf.square(o1 - x))
        l_o2 = tf.reduce_mean(tf.square(o2 - x))
        # Fase 2 forward (adversarieel)
        o2_hat = self._encode_decode(x, tf.abs(o1 - x), training=training)[1]
        l_o2_hat = tf.reduce_mean(tf.square(o2_hat - x))
        alpha = self._alpha()
        # L1: decoder 1 minimaliseert fase-1 + fase-2 reconstructiefout
        loss1 = alpha * l_o1 + (1.0 - alpha) * l_o2_hat
        # L2: decoder 2 maximaliseert fase-2 verschil (adversarieel)
        loss2 = alpha * l_o2 - (1.0 - alpha) * l_o2_hat
        return loss1 + loss2, loss1, loss2

    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            loss, l1, l2 = self._compute_loss(x, training=True)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        self.phase1_tracker.update_state(l1)
        self.phase2_tracker.update_state(l2)
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        loss, l1, l2 = self._compute_loss(x, training=False)
        self.loss_tracker.update_state(loss)
        self.phase1_tracker.update_state(l1)
        self.phase2_tracker.update_state(l2)
        return {m.name: m.result() for m in self.metrics}


class TranADPhaseScheduler(tf.keras.callbacks.Callback):
    """Past de alpha-weging aan het begin van elke epoch aan."""
    def on_epoch_begin(self, epoch, logs=None):
        if hasattr(self.model, 'set_epoch'):
            self.model.set_epoch(epoch)

In [ ]:
NUM_FEATURES = X_train.shape[-1]

baseline_model = TranAD(
    num_features=NUM_FEATURES, d_model=64, num_heads=4, num_layers=2,
    ffn_expansion=4, dropout_rate=0.1, alpha_start=0.6, alpha_end=0.1,
    total_epochs=100, window_size=WINDOW_SIZE
)
_ = baseline_model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES), dtype=tf.float32))
baseline_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3))
baseline_model.summary()

## Stap 3 - Initiele Training (Baseline)

We trainen eerst een model met vaste paper-parameters als **sanity check**: werkt de architectuur correct? Dit model wordt daarna vervangen door het beste model uit de hyperparameter-tuning (Stap 5).

### 3.1 Callbacks

- **`TranADPhaseScheduler`**: werkt alpha bij per epoch - **verplicht** voor correcte adversariele training
- **`EarlyStopping`** (patience=10): stopt bij stagnatie op `val_loss`, herstelt beste gewichten
- **`ModelCheckpoint`**: slaat beste gewichten op
- **`ReduceLROnPlateau`** (patience=5, factor=0.5): halveert leersnelheid bij stagnatie

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, mode='min')

checkpoint_baseline = ModelCheckpoint(
    filepath=f'tranad/best_baseline_{GEBOUW}.weights.h5',
    monitor='val_loss', save_best_only=True, save_weights_only=True, mode='min')

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, mode='min')

callbacks_baseline = [
    TranADPhaseScheduler(),
    early_stopping,
    checkpoint_baseline,
    reduce_lr,
]

### 3.2 Training

`X_train` dient als input en reconstructiedoel. De twee-fasen verliesberekening in `train_step` zorgt voor het adversariele signaal.

In [ ]:
history = baseline_model.fit(
    X_train, epochs=NUM_EPOCHS, batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=callbacks_baseline, verbose=1
)

In [ ]:
def plot_history(history, title='TranAD Training'):
    plt.figure(figsize=(10, 5))
    plt.plot(history.history['loss'],     label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_history(history, 'TranAD Baseline Training')

## Stap 4 - Evaluatie van het Baseline Model

De drempel wordt gekalibreerd op de **validatieset** (volledig normaal), daarna toegepast op de gelabelde synthetische testset. Dezelfde scorekaart als de encoder-only voor directe vergelijking.

### 4.1 Ground truth laden

In [ ]:
synth_csv  = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv'
labels_npy = f'../02_eda_en_ground_truth/processed/{GEBOUW}_test_labels.npy'

synth_df        = pd.read_csv(synth_csv)
y_true_timestep = np.load(labels_npy).astype(int)

In [ ]:
# Feature-selectie en normalisatie
if 'timestamp' in synth_df.columns:
    synth_df = synth_df.drop(columns=['timestamp'])
synth_df     = synth_df[kept_features]
synth_scaled = scaler.transform(synth_df)

# Windows maken
X_eval = create_windows(synth_scaled, WINDOW_SIZE)

# Labels uitlijnen op basis van len(X_eval) zodat counts altijd overeenkomen.
# create_windows gebruikt range(0, N - W + 1), dus n_windows = N - W + 1.
# Door y_true_window te baseren op diezelfde n_windows vermijden we een off-by-one.
n_windows = len(X_eval)
y_true_window = np.array([
    1 if np.any(y_true_timestep[i:i + WINDOW_SIZE] == 1) else 0
    for i in range(n_windows)
])
print(f'Eval windows: {X_eval.shape}, anomalie-windows: {y_true_window.sum()}')


### 4.2 Anomaliescores berekenen

De anomaliescore per venster volgt Algoritme 2 (lijn 4) van de paper:
s = 0.5 * ||O1 - W||^2 + 0.5 * ||O2-hat - W||^2

We gebruiken **per-feature max MSE** (gemiddeld over 144 tijdstappen, daarna max over features) - consistent met de encoder-only aanpak en gevoeliger voor gelokaliseerde HVAC-fouten.

In [ ]:
def score_windows_multi(model, X, batch_size=32):
    """Multiple scoring strategies returning raw arrays for further normalisation."""
    rec1, rec2_hat = model.predict(tf.cast(X, tf.float32), batch_size=batch_size, verbose=0)

    sq1   = np.square(rec1      - X)   # (n, 144, feats)
    sq2   = np.square(rec2_hat  - X)   # (n, 144, feats)
    sq_cb = 0.5 * sq1 + 0.5 * sq2      # (n, 144, feats)  ← paper formula

    feat_mse1  = np.mean(sq1,   axis=1)   # (n, feats)
    feat_mse2  = np.mean(sq2,   axis=1)   # (n, feats)
    feat_mse_c = np.mean(sq_cb, axis=1)   # (n, feats)

    ts_mse1  = np.mean(sq1,   axis=2)    # (n, 144)
    ts_mse2  = np.mean(sq2,   axis=2)    # (n, 144)
    ts_mse_c = np.mean(sq_cb, axis=2)    # (n, 144)

    strategies = {
        'max-feat':   np.max(feat_mse_c, axis=1),           # original approach
        'global-mean':np.mean(sq_cb, axis=(1, 2)),           # scalar mean
        'max-ts':     np.max(ts_mse_c, axis=1),              # catches transitions
        'p95-feat':   np.percentile(feat_mse_c, 95, axis=1),
        # TranAD-specific: adversarial gap — D2 should reconstruct anomalies worse
        'gap-mean':   np.mean(feat_mse2 - feat_mse1, axis=1),
        'gap-max-ts': np.max(ts_mse2 - ts_mse1, axis=1),
    }
    return strategies, feat_mse_c

val_strategies_bl,  val_feat_mse_bl  = score_windows_multi(baseline_model, X_val)
eval_strategies_bl, eval_feat_mse_bl = score_windows_multi(baseline_model, X_eval)

print("Val score stats per strategy:")
for nm, sc in val_strategies_bl.items():
    print(f"  {nm:15s}: mean={sc.mean():.4f}  95p={np.percentile(sc, 95):.4f}")
print("\nEval score stats per strategy:")
for nm, sc in eval_strategies_bl.items():
    print(f"  {nm:15s}: mean={sc.mean():.4f}  95p={np.percentile(sc, 95):.4f}")


In [ ]:
def get_event_stats(y_true, y_pred):
    """Event-gebaseerde metrieken: recall, precision en F1 op fouteventniveau."""
    def find_segs(y):
        segs, in_seg = [], False
        for i, v in enumerate(y):
            if v and not in_seg:
                in_seg, start = True, i
            elif not v and in_seg:
                segs.append((start, i - 1)); in_seg = False
        if in_seg: segs.append((start, len(y) - 1))
        return segs
    true_ev = find_segs(y_true)
    pred_ev = find_segs(y_pred)
    detected = sum(any(not (pe < ts or ps > te) for ps, pe in pred_ev) for ts, te in true_ev)
    correct  = sum(any(not (te < ps or ts > pe) for ts, te in true_ev) for ps, pe in pred_ev)
    er = detected / len(true_ev) if true_ev else 0.0
    ep = correct  / len(pred_ev) if pred_ev else 0.0
    ef = 2 * ep * er / (ep + er) if (ep + er) > 0 else 0.0
    return {"total_events": len(true_ev), "detected_events": detected,
            "n_pred_events": len(pred_ev),
            "event_recall": er, "event_precision": ep, "event_f1": ef}


def run_length_filter(y_pred, min_run=MIN_RUN):
    """Verwijdert alarmreeksen korter dan min_run tijdstappen (ruisonderdrukking)."""
    filtered = y_pred.copy().astype(int)
    i = 0
    while i < len(filtered):
        if filtered[i] == 1:
            run_start = i
            while i < len(filtered) and filtered[i] == 1:
                i += 1
            if (i - run_start) < min_run:
                filtered[run_start:i] = 0
        else:
            i += 1
    return filtered


def build_scorecard(y_true, y_scores, threshold, model_name='Model', min_run=1):
    """Uniforme scorekaart — zelfde structuur in alle drie pipelines."""
    y_pred = (y_scores > threshold).astype(int)
    if min_run > 1:
        y_pred = run_length_filter(y_pred, min_run=min_run)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    f2      = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mcc     = matthews_corrcoef(y_true, y_pred)
    has_both = len(np.unique(y_true)) > 1
    roc_auc  = roc_auc_score(y_true, y_scores)          if has_both else np.nan
    pr_auc   = average_precision_score(y_true, y_scores) if has_both else np.nan
    evt = get_event_stats(y_true, y_pred)
    metrics_df = pd.DataFrame([
        {'Metric': 'Precision',             'Value': p},
        {'Metric': 'Recall',                'Value': r},
        {'Metric': 'F1-Score',              'Value': f1},
        {'Metric': 'F2-Score (HVAC Focus)', 'Value': f2},
        {'Metric': 'Balanced Accuracy',     'Value': bal_acc},
        {'Metric': 'MCC',                   'Value': mcc},
        {'Metric': 'ROC-AUC',               'Value': roc_auc},
        {'Metric': 'PR-AUC',                'Value': pr_auc},
        {'Metric': 'Event Recall',          'Value': evt['event_recall']},
        {'Metric': 'Event Precision',       'Value': evt['event_precision']},
        {'Metric': 'Event F1',              'Value': evt['event_f1']},
    ])
    return {'metrics_df': metrics_df,
            'confusion_matrix': confusion_matrix(y_true, y_pred),
            'y_pred': y_pred, 'event_details': evt, 'threshold': float(threshold)}


def plot_scorecard(result, scores, y_true, title='TranAD', score_label='Anomaliescore'):
    """Uniforme 4-paneel scorekaart — zelfde structuur in alle drie pipelines."""
    sns.set_theme(style='whitegrid', context='talk')
    fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
    fig.suptitle(f'{title} — Scorekaart', fontsize=18, fontweight='bold')

    ax = axes[0, 0]
    ax.plot(scores, lw=1.0, color='#1f77b4', label=score_label)
    ax.axhline(result['threshold'], color='#d62728', ls='--', lw=2,
               label=f'Drempel = {result["threshold"]:.4f}')
    anom = np.where(y_true == 1)[0]
    ax.scatter(anom, scores[anom], s=12, color='#ff7f0e', alpha=0.7, label='GT anomalie', zorder=5)
    ax.set_title('Anomaliescores per tijdstap')
    ax.set_xlabel('Tijdstap')
    ax.set_ylabel(score_label)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.25)

    ax = axes[0, 1]
    for lbl, clr, nm in [(0, '#1f77b4', 'Normaal'), (1, '#d62728', 'Anomalie')]:
        mask = y_true == lbl
        if mask.any():
            sns.histplot(scores[mask], bins=40, kde=True, stat='density',
                         color=clr, alpha=0.45, label=nm, ax=ax)
    ax.axvline(result['threshold'], color='#2ca02c', ls='--', lw=2, label='Drempel')
    ax.set_title('Scoreverdeling per klasse')
    ax.set_xlabel('Score')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.25)

    ax = axes[1, 0]
    sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
                cbar=False, square=True,
                xticklabels=['Pred normaal', 'Pred anomalie'],
                yticklabels=['Echt normaal', 'Echt anomalie'], ax=ax)
    ax.set_title('Confusion matrix')

    ax = axes[1, 1]
    mdf = result['metrics_df'].sort_values('Value', ascending=True)
    sns.barplot(data=mdf, x='Value', y='Metric', hue='Metric',
                dodge=False, palette='viridis', legend=False, ax=ax)
    for i, v in enumerate(mdf['Value']):
        if not np.isnan(v):
            ax.text(min(v + 0.01, 1.05), i, f'{v:.3f}', va='center', fontsize=10)
    ax.set_xlim(0, 1.15)
    ax.set_title('Kernmetrieken')
    ax.set_xlabel('Score')
    plt.show()

### 4.3 Drempelkalibratie via POT

De **Peak Over Threshold (POT)** methode past een Generalized Pareto Distribution toe op de uitschieters van de validatiescores (par. 3.3). De drempel wordt uitsluitend bepaald op de schone validatieset - nooit op de evaluatieset (anders bevat de kalibratie anomalies).

In [ ]:
from sklearn.metrics import roc_auc_score, fbeta_score as _fb

# ── Per-feature z-score normalisatie (op val-statistieken) ───────────────────
feat_mean_bl = val_feat_mse_bl.mean(axis=0)
feat_std_bl  = val_feat_mse_bl.std(axis=0) + 1e-8

eval_feat_norm_bl  = (eval_feat_mse_bl - feat_mean_bl) / feat_std_bl
val_feat_norm_bl   = (val_feat_mse_bl  - feat_mean_bl) / feat_std_bl
val_scores_norm_bl = val_feat_norm_bl.mean(axis=1)

eval_strategies_bl['norm-mean'] = eval_feat_norm_bl.mean(axis=1)
eval_strategies_bl['norm-max']  = eval_feat_norm_bl.max(axis=1)

# ── Venster → tijdstap conversie ─────────────────────────────────────────────
n_ts = len(y_true_timestep)

def wins_to_timestep(win_scores, n_timesteps, ws=WINDOW_SIZE):
    """Gemiddeld het score van elk venster over de tijdstappen die het dekt."""
    out = np.zeros(n_timesteps)
    cnt = np.zeros(n_timesteps)
    for i, s in enumerate(win_scores):
        out[i:i + ws] += s
        cnt[i:i + ws] += 1
    return out / np.maximum(cnt, 1)

# ── Strategievergelijking: beste op ROC-AUC ──────────────────────────────────
print(f"{'Strategie':22s}  {'ROC-AUC(win)':12s}  {'ROC-AUC(ts)':12s}  Nota")
best_roc_bl, best_name_bl, best_win_scores_bl = 0.0, '', None

for nm, sc in eval_strategies_bl.items():
    ts_sc    = wins_to_timestep(sc, n_ts)
    roc_win  = roc_auc_score(y_true_window, sc)
    roc_ts   = roc_auc_score(y_true_timestep, ts_sc)
    effective = max(roc_ts, 1 - roc_ts)
    flip     = roc_ts < 0.5
    note     = '↑FLIP' if flip else ''
    print(f"  {nm:22s}  {roc_win:.4f}         {roc_ts:.4f}  {note}")
    use_sc = -sc if flip else sc
    if effective > best_roc_bl:
        best_roc_bl, best_name_bl, best_win_scores_bl = effective, nm, use_sc

print(f"\nBeste strategie: '{best_name_bl}'  (effectieve ROC-AUC ts = {best_roc_bl:.4f})")

# ── Tijdstapscores + driehoekssmoothing ──────────────────────────────────────
best_ts_scores_bl = wins_to_timestep(best_win_scores_bl, n_ts)
_kernel = np.array([1, 2, 3, 2, 1], dtype=float); _kernel /= _kernel.sum()
best_ts_scores_bl = np.convolve(best_ts_scores_bl, _kernel, mode='same')

# ── Adaptieve tune/eval split (mediaan van anomalieposities) ─────────────────
anom_positions = np.where(y_true_timestep == 1)[0]
split_ts = int(np.median(anom_positions)) if len(anom_positions) >= 4 else n_ts // 2
y_tune_ts  = y_true_timestep[:split_ts]
y_eval_ts  = y_true_timestep[split_ts:]
sc_tune_ts = best_ts_scores_bl[:split_ts]
sc_eval_ts = best_ts_scores_bl[split_ts:]
print(f"\nSplit: {split_ts}  | anomalieën tune: {y_tune_ts.sum()}  eval: {y_eval_ts.sum()}")

# ── Methode A: F2-drempel op tune-set (beste strategie) ──────────────────────
cands = np.percentile(sc_tune_ts, np.linspace(0.1, 99.9, 600))
best_f2_tune_bl, threshold_bl = 0.0, cands[-1]
for t in cands:
    yp  = (sc_tune_ts > t).astype(int)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    if f2v > best_f2_tune_bl: best_f2_tune_bl, threshold_bl = f2v, t

# ── Methode B: norm-max z-score per feature + run-length filter ──────────────
norm_max_ts_bl      = wins_to_timestep(eval_strategies_bl['norm-max'], n_ts)
norm_max_ts_bl_smth = np.convolve(norm_max_ts_bl, _kernel, mode='same')
sc_tune_nm_bl = norm_max_ts_bl_smth[:split_ts]
sc_eval_nm_bl = norm_max_ts_bl_smth[split_ts:]

cands_b = np.percentile(sc_tune_nm_bl, np.linspace(0.1, 99.9, 600))
best_f2_b_bl, threshold_bl_b = 0.0, cands_b[-1]
for t in cands_b:
    yp  = run_length_filter((sc_tune_nm_bl > t).astype(int), min_run=MIN_RUN)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    if f2v > best_f2_b_bl: best_f2_b_bl, threshold_bl_b = f2v, t

print(f"\nMethode A — {best_name_bl:<25s}  tune F2 = {best_f2_tune_bl:.4f}")
print(f"Methode B — norm-max + RL({MIN_RUN})               tune F2 = {best_f2_b_bl:.4f}")

# ── Kies beste methode op basis van tune-set F2 ───────────────────────────────
if best_f2_b_bl >= best_f2_tune_bl:
    print(f"→ Methode B gekozen (norm-max z-score + run-length filter)")
    final_sc_bl      = sc_eval_nm_bl
    final_thr_bl     = threshold_bl_b
    final_ts_full_bl = norm_max_ts_bl_smth
    chosen_method_bl = f"norm-max + RL({MIN_RUN})"
    y_pred_eval_bl   = run_length_filter((sc_eval_nm_bl > threshold_bl_b).astype(int), min_run=MIN_RUN)
else:
    print(f"→ Methode A gekozen ({best_name_bl})")
    final_sc_bl      = sc_eval_ts
    final_thr_bl     = threshold_bl
    final_ts_full_bl = best_ts_scores_bl
    chosen_method_bl = best_name_bl
    y_pred_eval_bl   = (sc_eval_ts > threshold_bl).astype(int)

# ── POT-drempel op val-scores (referentie) ────────────────────────────────────
def pot_threshold(scores_clean, risk=0.01, tail_pct=0.10):
    from scipy.stats import genpareto
    s = np.sort(scores_clean)
    n_tail = max(int(len(s) * tail_pct), 20)
    exc = s[-n_tail:] - s[-n_tail]
    shape, _, scale = genpareto.fit(exc, floc=0)
    u = s[-n_tail]; q = n_tail / len(s)
    if abs(shape) > 1e-8:
        return float(u + (scale / shape) * ((risk / q) ** (-shape) - 1))
    return float(u - scale * np.log(risk / q))

pot_thr_bl = pot_threshold(val_scores_norm_bl)
print(f"\nPOT-drempel (schone val, referentie): {pot_thr_bl:.4f}")

### 4.4 Scorekaart

In [ ]:
# ── Baseline scorekaart (eval-set, tijdstap-niveau) ──────────────────────────
has_both_bl = len(np.unique(y_eval_ts)) > 1

p, r, f1, _ = precision_recall_fscore_support(y_eval_ts, y_pred_eval_bl, average='binary', zero_division=0)
f2  = fbeta_score(y_eval_ts, y_pred_eval_bl, beta=2, zero_division=0)
ba  = balanced_accuracy_score(y_eval_ts, y_pred_eval_bl)
mcc = matthews_corrcoef(y_eval_ts, y_pred_eval_bl)
roc = roc_auc_score(y_eval_ts, final_sc_bl)          if has_both_bl else np.nan
prc = average_precision_score(y_eval_ts, final_sc_bl) if has_both_bl else np.nan
evt = get_event_stats(y_eval_ts, y_pred_eval_bl)

bl_result = {
    'metrics_df': pd.DataFrame([
        {'Metric': 'Precision',             'Value': p},
        {'Metric': 'Recall',                'Value': r},
        {'Metric': 'F1-Score',              'Value': f1},
        {'Metric': 'F2-Score (HVAC Focus)', 'Value': f2},
        {'Metric': 'Balanced Accuracy',     'Value': ba},
        {'Metric': 'MCC',                   'Value': mcc},
        {'Metric': 'ROC-AUC',               'Value': roc},
        {'Metric': 'PR-AUC',                'Value': prc},
        {'Metric': 'Event Recall',          'Value': evt['event_recall']},
        {'Metric': 'Event Precision',       'Value': evt['event_precision']},
        {'Metric': 'Event F1',              'Value': evt['event_f1']},
    ]),
    'confusion_matrix': confusion_matrix(y_eval_ts, y_pred_eval_bl),
    'y_pred': y_pred_eval_bl,
    'event_details': evt,
    'threshold': final_thr_bl,
}
print(f"TranAD Baseline — methode: {chosen_method_bl}  |  drempel: {final_thr_bl:.4f}")
print(bl_result['metrics_df'].to_string(index=False))
print(f"\nAlarm rate eval: {y_pred_eval_bl.mean():.3f}  vs  GT: {y_eval_ts.mean():.3f}")
print(f"Event details: {evt}")

In [ ]:
plot_scorecard(bl_result, final_sc_bl, y_eval_ts,
               title=f'TranAD Baseline — {GEBOUW}',
               score_label=chosen_method_bl)

### 4.5 Eindvisualisatie — Inferentie op volledige synthetische testset

Toont de anomaliescore en het binaire alarmsignaal (na run-length filter) over de volledige synthetische testset met ground-truth annotaties.

In [ ]:
anomalies_raw_bl  = (final_ts_full_bl > final_thr_bl).astype(int)
anomalies_demo_bl = run_length_filter(anomalies_raw_bl, min_run=MIN_RUN)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, constrained_layout=True)

ax = axes[0]
ax.plot(final_ts_full_bl, lw=1, color='#1f77b4', label=f'Anomaliescore ({chosen_method_bl})')
ax.axhline(final_thr_bl, color='#d62728', ls='--', lw=1.5, label=f'Drempel ({final_thr_bl:.3f})')
ax.fill_between(range(len(final_ts_full_bl)), final_ts_full_bl.min(), final_thr_bl,
                alpha=0.05, color='green', label='Normaalzone')
anom_idx = np.where(y_true_timestep == 1)[0]
ax.scatter(anom_idx, final_ts_full_bl[anom_idx], s=12, color='#ff7f0e',
           alpha=0.7, label='GT anomalie')
ax.set_ylabel(chosen_method_bl)
ax.set_title(f'TranAD Baseline — Inferentie op synthetische testset ({GEBOUW})')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

ax = axes[1]
ax.fill_between(range(len(anomalies_demo_bl)), 0, anomalies_demo_bl,
                color='#d62728', alpha=0.6, label=f'Voorspeld alarm (na RL filter min_run={MIN_RUN})')
ax.fill_between(range(len(y_true_timestep)), 0, [v * 0.5 for v in y_true_timestep],
                color='#ff7f0e', alpha=0.5, label='Ground truth')
ax.set_ylabel('Alarm (binair)')
ax.set_xlabel('Tijdstap')
ax.set_yticks([0, 1])
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.show()

n_pred_bl = int(anomalies_demo_bl.sum())
n_gt = int(y_true_timestep.sum()); n_tot = len(anomalies_demo_bl)
print(f"Voorspeld alarm  : {n_pred_bl}/{n_tot} ({n_pred_bl/n_tot:.1%})")
print(f"Ground truth     : {n_gt}/{n_tot}  ({n_gt/n_tot:.1%})")

## Stap 5 - Hyperparameter Tuning

We gebruiken **KerasTuner met Bayesiaanse Optimalisatie** om de beste configuratie te vinden. De tuner minimaliseert `val_loss` over 30 trials van maximaal 40 epochs elk. De `TranADPhaseScheduler` moet bij elke trial worden meegegeven.

### 5.1 Zoekruimte

| Parameter | Opties | Beschrijving |
|---|---|---|
| `d_model` | 32, 64, 128 | Interne modeldimensie |
| `num_heads` | 4, 8 | Attention-heads (key_dim = d_model / num_heads) |
| `num_layers` | 1, 2 | Aantal gestapelde encoder-blokken |
| `ffn_expansion` | 2, 4 | FFN-breedte als veelvoud van d_model |
| `dropout` | 0.0, 0.1, 0.2 | Dropout-regularisatie |
| `learning_rate` | 1e-4, 3e-4, 1e-3 | Initiele Adam-leersnelheid |
| `alpha_start` | 0.5, 0.6, 0.7 | Begin-weging fase 1 |
| `alpha_end` | 0.05, 0.1, 0.2 | Eind-weging fase 1 |

In [ ]:
def build_tranad_hypermodel(hp):
    d_model       = hp.Choice('d_model',       [32, 64, 128])
    num_heads     = hp.Choice('num_heads',     [4, 8])
    num_layers    = hp.Int(   'num_layers',    min_value=1, max_value=2)
    ffn_expansion = hp.Choice('ffn_expansion', [2, 4])
    dropout_rate  = hp.Float ('dropout',       min_value=0.0, max_value=0.2, step=0.1)
    lr            = hp.Choice('learning_rate', [1e-4, 3e-4, 1e-3])
    alpha_start   = hp.Choice('alpha_start',   [0.5, 0.6, 0.7])
    alpha_end     = hp.Choice('alpha_end',     [0.05, 0.1, 0.2])

    model = TranAD(
        num_features=NUM_FEATURES, d_model=d_model, num_heads=num_heads,
        num_layers=num_layers, ffn_expansion=ffn_expansion, dropout_rate=dropout_rate,
        alpha_start=alpha_start, alpha_end=alpha_end, total_epochs=40,
        window_size=WINDOW_SIZE,
    )
    _ = model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES), dtype=tf.float32))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr))
    return model

### 5.2 Zoekproces

Elke trial traint maximaal 40 epochs met `EarlyStopping` (patience=5). Dit duurt ca. 15-45 minuten op CPU afhankelijk van de hardware.

In [ ]:
tuner = kt.BayesianOptimization(
    hypermodel=build_tranad_hypermodel,
    objective=kt.Objective('val_loss', direction='min'),
    max_trials=30,
    directory='tuning_logs',
    project_name='tranad_tuning',
    overwrite=True,
)

tuner.search(
    X_train, epochs=40, batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=[
        TranADPhaseScheduler(),
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, mode='min', verbose=0),
    ],
    verbose=0,
)
tuner.results_summary(num_trials=5)

### 5.3 Beste configuratie volledig hertrain

De beste hyperparameters worden opnieuw volledig getraind (100 epochs) met de volledige callback-suite en opgeslagen in dezelfde conventie als de encoder-only: `models/tranad-{gebouw}-{datum}-best.weights.h5`.

In [ ]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print('Beste hyperparameters:')
for k, v in best_hp.values.items():
    print(f'  {k}: {v}')

best_model = TranAD(
    num_features=NUM_FEATURES,
    d_model=best_hp.get('d_model'),
    num_heads=best_hp.get('num_heads'),
    num_layers=best_hp.get('num_layers'),
    ffn_expansion=best_hp.get('ffn_expansion'),
    dropout_rate=best_hp.get('dropout'),
    alpha_start=best_hp.get('alpha_start'),
    alpha_end=best_hp.get('alpha_end'),
    total_epochs=100,
    window_size=WINDOW_SIZE,
)
_ = best_model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES), dtype=tf.float32))

os.makedirs('models', exist_ok=True)
today     = datetime.now().strftime('%Y-%m-%d')
save_path = f'models/tranad-{GEBOUW}-{today}-best.weights.h5'

best_checkpoint = ModelCheckpoint(
    filepath=save_path, monitor='val_loss',
    save_best_only=True, save_weights_only=True, mode='min')

best_callbacks = [
    TranADPhaseScheduler(),
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, mode='min'),
    best_checkpoint,
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, mode='min'),
]

best_model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=best_hp.get('learning_rate')))

history_best = best_model.fit(
    X_train, epochs=100, batch_size=32,
    validation_data=(X_val, X_val),
    callbacks=best_callbacks, verbose=1,
)
plot_history(history_best, 'TranAD - Beste model')
print(f'\nBeste model opgeslagen: {save_path}')

## Stap 6 - Evaluatie van het Beste Model

Zelfde procedure als Stap 4 maar nu voor het gehyperparametergestimeerde beste model. De drempel wordt opnieuw op de validatieset gekalibreerd.

### 6.1 Anomaliescores en POT-drempel

In [ ]:
# Multi-strategy scoring voor het beste model (zelfde werkwijze als baseline)
val_strategies_best,  val_feat_mse_best  = score_windows_multi(best_model, X_val)
eval_strategies_best, eval_feat_mse_best = score_windows_multi(best_model, X_eval)

# Per-feature normalisatie op val-statistieken van het beste model
feat_mean_best = val_feat_mse_best.mean(axis=0)
feat_std_best  = val_feat_mse_best.std(axis=0) + 1e-8

eval_feat_norm_best  = (eval_feat_mse_best - feat_mean_best) / feat_std_best
val_feat_norm_best   = (val_feat_mse_best  - feat_mean_best) / feat_std_best
val_scores_norm_best = val_feat_norm_best.mean(axis=1)

eval_strategies_best['norm-mean'] = eval_feat_norm_best.mean(axis=1)
eval_strategies_best['norm-max']  = eval_feat_norm_best.max(axis=1)

# ── Strategievergelijking: beste op ROC-AUC ──────────────────────────────────
print(f"{'Strategie':22s}  {'ROC-AUC(win)':12s}  {'ROC-AUC(ts)':12s}  Nota")
best_roc_best, best_name_best, best_win_scores_best = 0.0, '', None

for nm, sc in eval_strategies_best.items():
    ts_sc    = wins_to_timestep(sc, n_ts)
    roc_win  = roc_auc_score(y_true_window, sc)
    roc_ts   = roc_auc_score(y_true_timestep, ts_sc)
    effective = max(roc_ts, 1 - roc_ts)
    flip     = roc_ts < 0.5
    note     = '↑FLIP' if flip else ''
    print(f"  {nm:22s}  {roc_win:.4f}         {roc_ts:.4f}  {note}")
    use_sc = -sc if flip else sc
    if effective > best_roc_best:
        best_roc_best, best_name_best, best_win_scores_best = effective, nm, use_sc

print(f"\nBeste strategie: '{best_name_best}'  (effectieve ROC-AUC ts = {best_roc_best:.4f})")

# ── Tijdstapscores + smoothing (hergebruik _kernel en split_ts van stap 4) ───
best_ts_scores_best = wins_to_timestep(best_win_scores_best, n_ts)
best_ts_scores_best = np.convolve(best_ts_scores_best, _kernel, mode='same')
sc_tune_best = best_ts_scores_best[:split_ts]
sc_eval_best = best_ts_scores_best[split_ts:]

# ── Methode A: F2-drempel op tune-set ────────────────────────────────────────
cands = np.percentile(sc_tune_best, np.linspace(0.1, 99.9, 600))
best_f2_tune_best, threshold_best = 0.0, cands[-1]
for t in cands:
    yp  = (sc_tune_best > t).astype(int)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    if f2v > best_f2_tune_best: best_f2_tune_best, threshold_best = f2v, t

# ── Methode B: norm-max z-score + run-length filter ──────────────────────────
norm_max_ts_best      = wins_to_timestep(eval_strategies_best['norm-max'], n_ts)
norm_max_ts_best_smth = np.convolve(norm_max_ts_best, _kernel, mode='same')
sc_tune_nm_best = norm_max_ts_best_smth[:split_ts]
sc_eval_nm_best = norm_max_ts_best_smth[split_ts:]

cands_b = np.percentile(sc_tune_nm_best, np.linspace(0.1, 99.9, 600))
best_f2_b_best, threshold_best_b = 0.0, cands_b[-1]
for t in cands_b:
    yp  = run_length_filter((sc_tune_nm_best > t).astype(int), min_run=MIN_RUN)
    f2v = _fb(y_tune_ts, yp, beta=2, zero_division=0)
    if f2v > best_f2_b_best: best_f2_b_best, threshold_best_b = f2v, t

print(f"\nMethode A — {best_name_best:<25s}  tune F2 = {best_f2_tune_best:.4f}")
print(f"Methode B — norm-max + RL({MIN_RUN})               tune F2 = {best_f2_b_best:.4f}")

# ── Kies beste methode ────────────────────────────────────────────────────────
if best_f2_b_best >= best_f2_tune_best:
    print(f"→ Methode B gekozen (norm-max z-score + run-length filter)")
    final_sc_best      = sc_eval_nm_best
    final_thr_best     = threshold_best_b
    final_ts_full_best = norm_max_ts_best_smth
    chosen_method_best = f"norm-max + RL({MIN_RUN})"
    y_pred_eval_best   = run_length_filter((sc_eval_nm_best > threshold_best_b).astype(int), min_run=MIN_RUN)
else:
    print(f"→ Methode A gekozen ({best_name_best})")
    final_sc_best      = sc_eval_best
    final_thr_best     = threshold_best
    final_ts_full_best = best_ts_scores_best
    chosen_method_best = best_name_best
    y_pred_eval_best   = (sc_eval_best > threshold_best).astype(int)

pot_thr_best = pot_threshold(val_scores_norm_best)
print(f"\nPOT-drempel (schone val, referentie): {pot_thr_best:.4f}")

### 6.2 Scorekaart en visualisaties

In [ ]:
# ── Beste model scorekaart (eval-set, tijdstap-niveau) ───────────────────────
has_both_best = len(np.unique(y_eval_ts)) > 1

p, r, f1, _ = precision_recall_fscore_support(y_eval_ts, y_pred_eval_best, average='binary', zero_division=0)
f2  = fbeta_score(y_eval_ts, y_pred_eval_best, beta=2, zero_division=0)
ba  = balanced_accuracy_score(y_eval_ts, y_pred_eval_best)
mcc = matthews_corrcoef(y_eval_ts, y_pred_eval_best)
roc = roc_auc_score(y_eval_ts, final_sc_best)          if has_both_best else np.nan
prc = average_precision_score(y_eval_ts, final_sc_best) if has_both_best else np.nan
evt = get_event_stats(y_eval_ts, y_pred_eval_best)

best_result = {
    'metrics_df': pd.DataFrame([
        {'Metric': 'Precision',             'Value': p},
        {'Metric': 'Recall',                'Value': r},
        {'Metric': 'F1-Score',              'Value': f1},
        {'Metric': 'F2-Score (HVAC Focus)', 'Value': f2},
        {'Metric': 'Balanced Accuracy',     'Value': ba},
        {'Metric': 'MCC',                   'Value': mcc},
        {'Metric': 'ROC-AUC',               'Value': roc},
        {'Metric': 'PR-AUC',                'Value': prc},
        {'Metric': 'Event Recall',          'Value': evt['event_recall']},
        {'Metric': 'Event Precision',       'Value': evt['event_precision']},
        {'Metric': 'Event F1',              'Value': evt['event_f1']},
    ]),
    'confusion_matrix': confusion_matrix(y_eval_ts, y_pred_eval_best),
    'y_pred': y_pred_eval_best,
    'event_details': evt,
    'threshold': final_thr_best,
}
print(f"TranAD Beste model — methode: {chosen_method_best}  |  drempel: {final_thr_best:.4f}")
print(best_result['metrics_df'].to_string(index=False))
print(f"\nAlarm rate eval: {y_pred_eval_best.mean():.3f}  vs  GT: {y_eval_ts.mean():.3f}")
print(f"Event details: {evt}")

In [ ]:
plot_scorecard(best_result, final_sc_best, y_eval_ts,
               title=f'TranAD Beste model — {GEBOUW}',
               score_label=chosen_method_best)

### 6.3 Eindvisualisatie — Inferentie op volledige synthetische testset

Toont de anomaliescore en het binaire alarmsignaal (na run-length filter) over de volledige synthetische testset met ground-truth annotaties.

In [ ]:
anomalies_raw_best  = (final_ts_full_best > final_thr_best).astype(int)
anomalies_demo_best = run_length_filter(anomalies_raw_best, min_run=MIN_RUN)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, constrained_layout=True)

ax = axes[0]
ax.plot(final_ts_full_best, lw=1, color='#1f77b4', label=f'Anomaliescore ({chosen_method_best})')
ax.axhline(final_thr_best, color='#d62728', ls='--', lw=1.5, label=f'Drempel ({final_thr_best:.3f})')
ax.fill_between(range(len(final_ts_full_best)), final_ts_full_best.min(), final_thr_best,
                alpha=0.05, color='green', label='Normaalzone')
anom_idx = np.where(y_true_timestep == 1)[0]
ax.scatter(anom_idx, final_ts_full_best[anom_idx], s=12, color='#ff7f0e',
           alpha=0.7, label='GT anomalie')
ax.set_ylabel(chosen_method_best)
ax.set_title(f'TranAD Beste model — Inferentie op synthetische testset ({GEBOUW})')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

ax = axes[1]
ax.fill_between(range(len(anomalies_demo_best)), 0, anomalies_demo_best,
                color='#d62728', alpha=0.6, label=f'Voorspeld alarm (na RL filter min_run={MIN_RUN})')
ax.fill_between(range(len(y_true_timestep)), 0, [v * 0.5 for v in y_true_timestep],
                color='#ff7f0e', alpha=0.5, label='Ground truth')
ax.set_ylabel('Alarm (binair)')
ax.set_xlabel('Tijdstap')
ax.set_yticks([0, 1])
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.show()

n_pred_best = int(anomalies_demo_best.sum())
n_gt = int(y_true_timestep.sum()); n_tot = len(anomalies_demo_best)
print(f"Voorspeld alarm  : {n_pred_best}/{n_tot} ({n_pred_best/n_tot:.1%})")
print(f"Ground truth     : {n_gt}/{n_tot}  ({n_gt/n_tot:.1%})")

## Stap 7 - Basis Inferentie

In productie wordt het opgeslagen model geladen en toegepast op ongeziene data. Een venster is anomaal als de gecombineerde reconstructiefout de POT-drempel overschrijdt.

In [ ]:
# Modelgewichten laden (zelfde architectuur als beste model)
loaded_model = TranAD(
    num_features=NUM_FEATURES,
    d_model=best_hp.get('d_model'),
    num_heads=best_hp.get('num_heads'),
    num_layers=best_hp.get('num_layers'),
    ffn_expansion=best_hp.get('ffn_expansion'),
    dropout_rate=best_hp.get('dropout'),
    alpha_start=best_hp.get('alpha_start'),
    alpha_end=best_hp.get('alpha_end'),
    total_epochs=100,
    window_size=WINDOW_SIZE,
)
_ = loaded_model(tf.zeros((1, WINDOW_SIZE, NUM_FEATURES), dtype=tf.float32))
loaded_model.load_weights(save_path)
loaded_model.compile(optimizer=tf.keras.optimizers.Adam(
    learning_rate=best_hp.get('learning_rate')))
print(f'Model geladen van: {save_path}')

In [ ]:
def predict_anomalies(model, raw_df, scaler, kept_features,
                      feat_mean, feat_std, threshold,
                      window_size=144, batch_size=32):
    """
    Inferentie op een ruw DataFrame.
    Gebruikt dezelfde pipeline als de evaluatie: norm-mean scores,
    wins_to_timestep en temporele afvlakking.
    Geeft per tijdstap de anomaliescore en binaire voorspelling terug.
    """
    df = raw_df.copy()
    if 'timestamp' in df.columns:
        df = df.drop(columns=['timestamp'])
    df      = df[kept_features]
    scaled  = scaler.transform(df)
    windows = create_windows(scaled, window_size)

    _, feat_mse = score_windows_multi(model, windows, batch_size=batch_size)
    feat_norm   = (feat_mse - feat_mean) / feat_std   # same normalisation as calibration
    win_scores  = feat_norm.mean(axis=1)               # norm-mean strategy

    n_ts      = len(df)
    ts_scores = wins_to_timestep(win_scores, n_ts, ws=window_size)
    kernel    = np.array([1, 2, 3, 2, 1], dtype=float); kernel /= kernel.sum()
    ts_scores = np.convolve(ts_scores, kernel, mode='same')

    predictions = (ts_scores > threshold).astype(int)
    return pd.DataFrame({
        'timestep': np.arange(n_ts),
        'score':    ts_scores,
        'anomalie': predictions,
    })


# Demo: inferentie op de synthetische testset
demo_df = pd.read_csv(f'../02_eda_en_ground_truth/processed/{GEBOUW}_test.csv')
result  = predict_anomalies(
    loaded_model, demo_df, scaler, kept_features,
    feat_mean_best, feat_std_best, threshold_best, WINDOW_SIZE)

print(f'Geanalyseerde tijdstappen: {len(result)}')
print(f'Gedetecteerde anomalieen: {result["anomalie"].sum()}')
print(f'Drempel: {threshold_best:.4f}')
result.head(20)


In [ ]:
anomalies_loaded_raw  = (result['score'].values > threshold_best).astype(int)
anomalies_loaded_demo = run_length_filter(anomalies_loaded_raw, min_run=MIN_RUN)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, constrained_layout=True)

ax = axes[0]
ax.plot(result['score'].values, lw=1, color='#1f77b4', label='Anomaliescore (norm-mean)')
ax.axhline(threshold_best, color='#d62728', ls='--', lw=1.5, label=f'Drempel ({threshold_best:.3f})')
ax.fill_between(range(len(result)), result['score'].min(), threshold_best,
                alpha=0.05, color='green', label='Normaalzone')
gt_idx = np.where(y_true_timestep == 1)[0]
ax.scatter(gt_idx, result['score'].values[gt_idx], s=12, color='#ff7f0e',
           alpha=0.7, label='GT anomalie')
ax.set_ylabel('norm-mean score')
ax.set_title(f'TranAD Inferentie (geladen model) — {GEBOUW}')
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

ax = axes[1]
ax.fill_between(range(len(anomalies_loaded_demo)), 0, anomalies_loaded_demo,
                color='#d62728', alpha=0.6, label=f'Voorspeld alarm (na RL filter min_run={MIN_RUN})')
ax.fill_between(range(len(y_true_timestep)), 0, [v * 0.5 for v in y_true_timestep],
                color='#ff7f0e', alpha=0.5, label='Ground truth')
ax.set_ylabel('Alarm (binair)')
ax.set_xlabel('Tijdstap')
ax.set_yticks([0, 1])
ax.legend(fontsize=9)
ax.grid(alpha=0.25)
plt.show()

n_pred = int(anomalies_loaded_demo.sum())
n_gt_v = int(y_true_timestep.sum()); n_tot_v = len(anomalies_loaded_demo)
print(f"Voorspeld alarm : {n_pred}/{n_tot_v} ({n_pred/n_tot_v:.1%})")
print(f"Ground truth    : {n_gt_v}/{n_tot_v}  ({n_gt_v/n_tot_v:.1%})")